# EduHinglish: Model Server on Colab T4 (for the backend)

Runs the trained model and exposes it as an HTTP API with a public URL. The project backend (`backend/`) calls this URL.

```
Frontend ──► backend/ (FastAPI, any machine) ──HTTPS + X-API-Key──► this notebook (T4 GPU)
                                                                    M1 intent → M2 ChromaDB → M3 Gemma + LoRA
```

**Secrets** (🔑 sidebar, enable notebook access):
| Name | What |
|---|---|
| `HF_TOKEN` | Hugging Face token with **read** access (needed for gated Gemma) |
| `MODEL_API_KEY` | any long random string; put the same value in `backend/.env` |
| `NGROK_AUTHTOKEN` | optional: gives a **fixed URL** (see the Tunnel cell) |
| `GITHUB_TOKEN` | only if the repo is private |

> **Runtime**: Runtime → Change runtime type → **T4 GPU**. Run the pip cell, **Restart session**, then **Run all**.
> Google Drive is used for the knowledge base cache (~48 s to build on first run, instant on reruns).

In [ ]:
!pip install -q -U "bitsandbytes>=0.43.0" "transformers>=4.47.0" "peft>=0.14.0" "accelerate>=1.0.0" \
    "chromadb>=0.4.24" "sentence-transformers>=2.7.0" nltk colorama \
    "fastapi>=0.111" "uvicorn>=0.29" pyngrok

## 1. Settings

In [10]:
GITHUB_REPO = "https://github.com/PersuasivePost/EduHinglish.git"
BRANCH      = "develop"

# ── Google Drive ─────────────────────────────────────────────────────────────
BASE_DIR = "/content/drive/MyDrive/EduHinglish"
KB_CACHE = f"{BASE_DIR}/src/knowledge_base"   # cached across sessions; ~48 s to build first time

# ── HuggingFace repos (fill in after pushing with scripts/push_to_hub.py) ──
HF_ADAPTER_REPO = "persuasivepost/eduhinglish-gemma-3-4b-it-lora"
HF_INTENT_REPO  = "persuasivepost/eduhinglish-muril-intent"  # set to None to skip

PORT = 8000
# Fixed URL with ngrok (free): create a static domain at https://dashboard.ngrok.com/domains
# and paste it here, e.g. "eduhinglish-demo.ngrok-free.app". Leave "" for a random ngrok URL.
# Without the NGROK_AUTHTOKEN secret, a Cloudflare quick tunnel is used (random URL, no account).
NGROK_DOMAIN = ""


## 2. Drive & secrets


In [11]:
import os, sys, shutil, subprocess
from google.colab import drive, userdata

drive.mount("/content/drive")

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
if not os.environ["HF_TOKEN"]:
    print("WARNING: no HF_TOKEN secret — Gemma is a gated model and will fail to download.")
os.environ["MODEL_API_KEY"] = secret("MODEL_API_KEY") or ""
if not os.environ["MODEL_API_KEY"]:
    print("WARNING: no MODEL_API_KEY secret, so anyone with the URL can use the GPU.")

gh = secret("GITHUB_TOKEN")
url = GITHUB_REPO.replace("https://", f"https://{gh}@") if gh else GITHUB_REPO
REPO = "/content/EduHinglish"
if os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
subprocess.run([sys.executable, "scripts/download_nltk_data.py"], check=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


CompletedProcess(args=['/usr/bin/python3', 'scripts/download_nltk_data.py'], returncode=0)

## 3. Knowledge base (~48 s, built from repo text)


In [12]:
KB_LOCAL = f"{REPO}/src/knowledge_base"

def kb_count(path):
    from retriever import NCERTRetriever
    try:
        return NCERTRetriever(db_path=path).get_collection_stats()["total_documents"]
    except Exception as e:
        print("KB load failed:", e)
        return 0

# Use Drive cache if available (skips ~48 s rebuild on reruns)
if kb_count(KB_CACHE) > 0:
    KB_PATH = KB_CACHE
    print("Knowledge base loaded from Drive:", kb_count(KB_PATH), "chunks")
else:
    if kb_count(KB_LOCAL) == 0:
        subprocess.run([sys.executable, "scripts/build_knowledge_base.py", "--force"], check=True)
    KB_PATH = KB_LOCAL
    if not os.path.isdir(KB_CACHE):
        shutil.copytree(KB_PATH, KB_CACHE)
        print("Knowledge base cached to Drive for next session:", KB_CACHE)
    print("Knowledge base chunks:", kb_count(KB_PATH))


  [OK] NCERTRetriever initialized (ChromaDB @ /content/drive/MyDrive/EduHinglish/src/knowledge_base)
       Collection:  ncert_textbooks
       Documents:   0
  [OK] NCERTRetriever initialized (ChromaDB @ /content/EduHinglish/src/knowledge_base)
       Collection:  ncert_textbooks
       Documents:   5007
  [OK] NCERTRetriever initialized (ChromaDB @ /content/EduHinglish/src/knowledge_base)
       Collection:  ncert_textbooks
       Documents:   5007
Knowledge base chunks: 5007


In [13]:
import os
token = os.environ.get("HF_TOKEN", "")
print(f"Token set: {bool(token)}, starts with: {token[:8] if token else 'EMPTY'}")

Token set: True, starts with: hf_OWCcF


## 4. Load the model (~3–5 min)

In [14]:
os.environ["EDUHINGLISH_ADAPTER"] = HF_ADAPTER_REPO
if HF_INTENT_REPO:
    os.environ["EDUHINGLISH_INTENT_MODEL"] = HF_INTENT_REPO
os.environ["EDUHINGLISH_KB_PATH"] = KB_PATH

from pipeline import EduHinglishPipeline
pipe = EduHinglishPipeline(mode="full", use_intent_model=bool(HF_INTENT_REPO))
print("LoRA loaded:", pipe.generator.lora_loaded, "|", pipe.generator.adapter_source)

  EduHinglish — Pipeline  v2.0  (mode=full)
[OK] spaCy en_core_web_sm loaded
[OK] EnglishPreprocessor initialized
[OK] HinglishNormalizer initialized
[OK] WordLevelLID initialized

  Loading embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  (First run will download ~500MB - subsequent runs use cache)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  [OK] NCERTEmbedder initialized
       Model:     paraphrase-multilingual-MiniLM-L12-v2
       Dimension: 384
       Device:    cuda:0
  [OK] NCERTRetriever initialized (ChromaDB @ /content/EduHinglish/src/knowledge_base)
       Collection:  ncert_textbooks
       Documents:   5007


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[WARN] Intent classifier not loaded (Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported).
       Gemma will answer without an intent hint.

  Loading Hinglish generator: google/gemma-3-4b-it


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

... (11 outputs left out)

adapter_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  131MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

  [OK] HinglishGenerator initialized
       Model:        Gemma-3-4b-it + LoRA
       Device:       cuda (4-bit)
       LoRA adapter: persuasivepost/eduhinglish-gemma-3-4b-it-lora

[OK] All modules loaded. Pipeline ready (mode=full).
LoRA loaded: True | persuasivepost/eduhinglish-gemma-3-4b-it-lora


## 5. Start the API server

In [15]:
import model_server, requests
server = model_server.start_in_thread(pipe, port=PORT)
print(requests.get(f"http://localhost:{PORT}/health").json())

[OK] Model server on http://0.0.0.0:8000
{'status': 'ok', 'mode': 'full', 'started_at': '2026-11-05 08:50:28', 'adapter': 'persuasivepost/eduhinglish-gemma-3-4b-it-lora', 'lora_loaded': True, 'device': 'cuda', 'intent_model': False, 'kb_chunks': 5007, 'busy': False}


## 6. Public URL (tunnel)

In [21]:
import re, time

ngrok_token = secret("NGROK_AUTHTOKEN")
if ngrok_token:
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token)
    kwargs = {"domain": NGROK_DOMAIN} if NGROK_DOMAIN else {}
    PUBLIC_URL = ngrok.connect(PORT, "http", **kwargs).public_url
else:
    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                       check=True)
        os.chmod("/usr/local/bin/cloudflared", 0o755)
    log = open("/content/cloudflared.log", "w")
    tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                              stdout=log, stderr=subprocess.STDOUT)
    PUBLIC_URL = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
        if m:
            PUBLIC_URL = m.group(0)
            break
        time.sleep(1)
    assert PUBLIC_URL, "cloudflared did not print a URL; see /content/cloudflared.log"

print("=" * 70)
print("MODEL_SERVER_URL =", PUBLIC_URL)
print("Put this in backend/.env as COLAB_URL=<url> and restart the backend.")
print("=" * 70)

# Save URL to Drive so the backend config can reference it
os.makedirs(BASE_DIR, exist_ok=True)
with open(f"{BASE_DIR}/model_server_url.txt", "w") as f:
    f.write(PUBLIC_URL + "\n")
print(f"URL saved to Drive: {BASE_DIR}/model_server_url.txt")


MODEL_SERVER_URL = https://grade-helpful-psychiatry-rouge.trycloudflare.com
Put this in backend/.env as COLAB_URL=<url> and restart the backend.
URL saved to Drive: /content/drive/MyDrive/EduHinglish/model_server_url.txt


## 7. Test through the public URL (same call the backend makes)

In [17]:
headers = {"X-API-Key": os.environ["MODEL_API_KEY"], "ngrok-skip-browser-warning": "1"}
print(requests.get(f"{PUBLIC_URL}/health", headers=headers, timeout=30).json())
r = requests.post(f"{PUBLIC_URL}/ask", headers=headers, timeout=180,
                  json={"query": "Ohm ka law kya hai?", "subject": "physics", "class_num": "10"})
print(r.status_code)
res = r.json()
print("ANSWER:", res.get("answer"))
print("INTENT:", res.get("intent"), res.get("intent_confidence"))
print("TIMINGS:", res.get("timings"))

{'status': 'ok', 'mode': 'full', 'started_at': '2026-11-05 08:50:28', 'adapter': 'persuasivepost/eduhinglish-gemma-3-4b-it-lora', 'lora_loaded': True, 'device': 'cuda', 'intent_model': False, 'kb_chunks': 5007, 'busy': False}

  EduHinglish — Full Pipeline (M1→M2→M3)
  Query: "Ohm ka law kya hai?"

  [STEP 1/3] M1 — Preprocessing
SCRIPT DETECTION
  Input : "Ohm ka law kya hai?"

  Word                         Script
  ---------------------------- ------
  Ohm                          ROMAN
  ka                           ROMAN
  law                          ROMAN
  kya                          ROMAN
  hai?                         ROMAN

  Distribution:
    Roman      :   5 words  (100.0%)
    Devanagari :   0 words  (0.0%)
    Mixed      :   0 words  (0.0%)

  → Overall script : ROMAN

[PIPELINE MODE] HINGLISH
  Input: "Ohm ka law kya hai?"

--- HINGLISH STEP 1 / 6 : Script Detection ---
SCRIPT DETECTION
  Input : "Ohm ka law kya hai?"

  Word                         Script
  ----------

## 8. Keep the server running
The server runs as long as this runtime is alive. This cell prints a heartbeat every 5 minutes.
Stop it (■) when the demo is over; that also shuts down the URL.

In [19]:
while True:
    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=10).json()
        print(time.strftime("%H:%M:%S"), "ok | busy:", h.get("busy"), "|", PUBLIC_URL)
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "health check failed:", e)
    time.sleep(300)

08:59:32 ok | busy: False | https://seconds-writings-event-arrive.trycloudflare.com


KeyboardInterrupt: 

In [20]:
import requests; print(requests.get(f"http://localhost:{PORT}/health").json())

{'status': 'ok', 'mode': 'full', 'started_at': '2026-11-05 08:50:28', 'adapter': 'persuasivepost/eduhinglish-gemma-3-4b-it-lora', 'lora_loaded': True, 'device': 'cuda', 'intent_model': False, 'kb_chunks': 5007, 'busy': False}
